# Exploração de Dados

In [ ]:
# %% [markdown]
# # 🔐 Análise de Vulnerabilidades CISA KEV
# 
# **Dados reais do catálogo de vulnerabilidades conhecidas e exploradas**
# 
# ## 📊 Contexto
# Este notebook analisa o catálogo **CISA Known Exploited Vulnerabilities (KEV)**, 
# que contém vulnerabilidades que estão sendo ativamente exploradas.

# %%
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.etl_pipeline import CisaKEVAnalyzer
from src.analytics import CisaAnalytics

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')

# %% [markdown]
# ## 1. Executar Pipeline ETL

# %%
# Executar pipeline completo
analyzer = CisaKEVAnalyzer()
analyzer.run_pipeline()

# %% [markdown]
# ## 2. Explorar Camada Silver

# %%
# Carregar dados processados
df = pd.read_parquet('../data/processed/silver_vulnerabilities.parquet')

print(f"🔍 Shape: {df.shape}")
print(f"\n📋 Colunas disponíveis:")
for col in df.columns:
    print(f"   - {col}")

# %%
# Primeiras linhas
df.head()

# %%
# Informações básicas
df.info()

# %% [markdown]
# ## 3. Estatísticas Descritivas

# %%
# Estatísticas das colunas numéricas
df[['cvss_score', 'days_to_due', 'days_since_added', 'cwe_count']].describe()

# %%
# Análise por nível de risco
df.groupby('risk_level').agg({
    'cve_id': 'count',
    'cvss_score': ['mean', 'max'],
    'vendor': 'nunique',
    'is_ransomware': 'sum'
}).round(2)

# %% [markdown]
# ## 4. Análise por Fabricante

# %%
# Top 20 fabricantes
top_vendors = df['vendor'].value_counts().head(20)
print("🏢 TOP 20 FABRICANTES MAIS AFETADOS:")
print(top_vendors)

# %%
# Análise detalhada de um fabricante específico
fabricante = 'Microsoft'
df_microsoft = df[df['vendor'] == fabricante]

print(f"📊 ANÁLISE {fabricante}:")
print(f"   Total de vulnerabilidades: {len(df_microsoft)}")
print(f"   Média CVSS: {df_microsoft['cvss_score'].mean():.1f}")
print(f"   Vulnerabilidades críticas: {df_microsoft['is_critical'].sum()}")
print(f"   Usadas em ransomware: {df_microsoft['is_ransomware'].sum()}")

# %% [markdown]
# ## 5. Análise Temporal

# %%
# Vulnerabilidades por ano
temporal = df['year_added'].value_counts().sort_index()
print("📅 VULNERABILIDADES POR ANO:")
print(temporal)

# %%
# Gráfico de tendência
plt.figure(figsize=(12, 6))
plt.plot(temporal.index, temporal.values, marker='o', linewidth=2, markersize=8)
plt.fill_between(temporal.index, temporal.values, alpha=0.3)
plt.title('Evolução de Vulnerabilidades por Ano', fontsize=14, fontweight='bold')
plt.xlabel('Ano')
plt.ylabel('Quantidade')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 6. Análise de Ransomware

# %%
# Vulnerabilidades usadas em ransomware
df_ransom = df[df['is_ransomware']]

print(f"💀 VULNERABILIDADES USADAS EM RANSOMWARE:")
print(f"   Total: {len(df_ransom)}")
print(f"   Percentual: {(len(df_ransom) / len(df) * 100):.1f}%")
print(f"\n   Por fabricante:")
print(df_ransom['vendor'].value_counts().head(10))

# %%
# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza
ransom_counts = df['ransomware_known'].value_counts()
axes[0].pie(ransom_counts.values, labels=ransom_counts.index, autopct='%1.1f%%',
            colors=['red', 'green'], explode=[0.05, 0])
axes[0].set_title('Uso em Ransomware', fontsize=12, fontweight='bold')

# Barras
ransom_vendor = df_ransom['vendor'].value_counts().head(8)
axes[1].barh(range(len(ransom_vendor)), ransom_vendor.values, color='crimson')
axes[1].set_yticks(range(len(ransom_vendor)))
axes[1].set_yticklabels(ransom_vendor.index)
axes[1].set_title('Top 8 Fabricantes em Ransomware', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Quantidade')

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 7. Análise de CWEs

# %%
# Top CWEs
cwes_exploded = df.assign(cwe=df['cwe_list'].str.split(',')).explode('cwe')
cwes_exploded['cwe'] = cwes_exploded['cwe'].str.strip()
top_cwes = cwes_exploded['cwe'].value_counts().head(15)

print("🔧 TOP 15 CWEs MAIS COMUNS:")
print(top_cwes)

# %%
# Gráfico de CWEs
plt.figure(figsize=(12, 6))
top_cwes.plot(kind='bar', color='teal', alpha=0.7)
plt.title('Top 15 CWEs Mais Frequentes', fontsize=14, fontweight='bold')
plt.xlabel('CWE')
plt.ylabel('Frequência')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 8. Insights Estratégicos

# %%
print("🔍 INSIGHTS ESTRATÉGICOS:")
print("="*50)

# Insight 1: Fabricantes críticos
top_critical = df[df['is_critical']]['vendor'].value_counts().head(3)
print(f"\n1️⃣ Fabricantes com mais vulnerabilidades CRÍTICAS:")
for vendor, count in top_critical.items():
    print(f"   - {vendor}: {count}")

# Insight 2: Tendência temporal
ultimo_ano = df['year_added'].max()
ano_anterior = ultimo_ano - 1
crescimento = ((df['year_added'] == ultimo_ano).sum() - 
               (df['year_added'] == ano_anterior).sum())
print(f"\n2️⃣ Tendência: {crescimento:+,d} vulnerabilidades de {ano_anterior} para {ultimo_ano}")

# Insight 3: Ransomware
print(f"\n3️⃣ Ransomware: {len(df_ransom)} vulnerabilidades ativamente usadas em ataques de ransomware")

# Insight 4: Urgência
urgentes = df[(df['days_to_due'] < 30) & (df['days_to_due'] > 0)]
print(f"\n4️⃣ Urgência: {len(urgentes)} vulnerabilidades com prazo < 30 dias para correção")

# Insight 5: CWEs mais perigosas
cwe_critical = cwes_exploded[cwes_exploded['is_critical']]['cwe'].value_counts().head(3)
print(f"\n5️⃣ CWEs mais associadas a vulnerabilidades críticas:")
for cwe, count in cwe_critical.items():
    print(f"   - {cwe}: {count}")

# %% [markdown]
# ## 9. Gerar Relatórios Finais

# %%
# Gerar todos os relatórios
analytics = CisaAnalytics()
analytics.generate_executive_report()
analytics.create_strategic_visualizations()
analytics.generate_ransomware_report()
analytics.generate_compliance_report()

print("\n✅ Todos os relatórios gerados com sucesso!")